# Preprocess Datasets

preprocess english + italian data

In [1]:
from pathlib import Path
import pandas as pd

from utils.preprocessing import SentenceSplitPreprocessor

In [2]:
RAW_ROOT = Path("./data/raw")
PROCESSED_ROOT = Path("./data/processed")

PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

prep = SentenceSplitPreprocessor(window=50)

In [3]:
raw_files = sorted(RAW_ROOT.glob("*/*.sent_split"))

print(f"total raw files found: {len(raw_files)}")
for path in raw_files:
    print(path)

total raw files found: 22
data\raw\UD_English-EWT\en_ewt-ud-dev.sent_split
data\raw\UD_English-EWT\en_ewt-ud-test.sent_split
data\raw\UD_English-EWT\en_ewt-ud-train.sent_split
data\raw\UD_English-GUM\en_gum-ud-dev.sent_split
data\raw\UD_English-GUM\en_gum-ud-test.sent_split
data\raw\UD_English-GUM\en_gum-ud-train.sent_split
data\raw\UD_English-ParTUT\en_partut-ud-dev.sent_split
data\raw\UD_English-ParTUT\en_partut-ud-test.sent_split
data\raw\UD_English-ParTUT\en_partut-ud-train.sent_split
data\raw\UD_English-PUD\en_pud-ud-test.sent_split
data\raw\UD_Italian-ISDT\it_isdt-ud-dev.sent_split
data\raw\UD_Italian-ISDT\it_isdt-ud-test.sent_split
data\raw\UD_Italian-ISDT\it_isdt-ud-train.sent_split
data\raw\UD_Italian-MarkIT\it_markit-ud-dev.sent_split
data\raw\UD_Italian-MarkIT\it_markit-ud-test.sent_split
data\raw\UD_Italian-MarkIT\it_markit-ud-train.sent_split
data\raw\UD_Italian-ParTUT\it_partut-ud-dev.sent_split
data\raw\UD_Italian-ParTUT\it_partut-ud-test.sent_split
data\raw\UD_Italian-P

In [6]:
#sanity check
sample_path = raw_files[0]

result = prep.process_file(sample_path)
df_sample = result["df"]

print("sample file:", sample_path)
print("shape:", df_sample.shape)
print(df_sample.head())
print(df_sample["label"].value_counts())

sample file: data\raw\UD_English-EWT\en_ewt-ud-dev.sent_split
shape: (2294, 9)
          doc_id  char_idx punct  \
0  en_ewt-ud-dev       154     .   
1  en_ewt-ud-dev       182     .   
2  en_ewt-ud-dev       301     .   
3  en_ewt-ud-dev       308     .   
4  en_ewt-ud-dev       337     .   

                                        left_context  \
0  g jurists\non federal courts in the Washington...   
1  in the Washington area.\n Bush nominated Jenni...   
2  t of the District of\nColumbia, replacing Stef...   
3  e District of\nColumbia, replacing Steffen W. ...   
4  cing Steffen W. Graae.\n ***\n Bush also nomin...   

                                       right_context  \
0  \n Bush nominated Jennifer M. Anderson\nfor a ...   
1   Anderson\nfor a 15-year term as associate jud...   
2   Graae.\n ***\n Bush also nominated A. Noel An...   
3  \n ***\n Bush also nominated A. Noel Anketell\...   
4   Noel Anketell\nKramer for a 15-year term as a...   

                              

In [5]:
def make_output_path(raw_path: Path, processed_root: Path) -> Path:
    dataset_name = raw_path.parent.name
    out_dir = processed_root / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    out_name = raw_path.stem + ".csv"
    return out_dir / out_name

In [7]:
summary_rows = []

for raw_path in raw_files:
    result = prep.process_file(raw_path)
    df = result["df"]

    out_path = make_output_path(raw_path, PROCESSED_ROOT)
    df.to_csv(out_path, index=False, encoding="utf-8")

    summary_rows.append({
        "dataset": raw_path.parent.name,
        "raw_file": raw_path.name,
        "processed_file": out_path.name,
        "n_rows": len(df),
        "n_positive": int(df["label"].sum()),
        "n_negative": int((df["label"] == 0).sum()),
        "clean_text_len": len(result["clean_text"]),
        "newline_only_boundaries": len(result["newline_boundary_indices"]),
    })

    print(f"saved: {out_path} | rows={len(df)}")

saved: data\processed\UD_English-EWT\en_ewt-ud-dev.csv | rows=2294
saved: data\processed\UD_English-EWT\en_ewt-ud-test.csv | rows=2083
saved: data\processed\UD_English-EWT\en_ewt-ud-train.csv | rows=13649
saved: data\processed\UD_English-GUM\en_gum-ud-dev.csv | rows=1568
saved: data\processed\UD_English-GUM\en_gum-ud-test.csv | rows=1420
saved: data\processed\UD_English-GUM\en_gum-ud-train.csv | rows=10020
saved: data\processed\UD_English-ParTUT\en_partut-ud-dev.csv | rows=153
saved: data\processed\UD_English-ParTUT\en_partut-ud-test.csv | rows=152
saved: data\processed\UD_English-ParTUT\en_partut-ud-train.csv | rows=1724
saved: data\processed\UD_English-PUD\en_pud-ud-test.csv | rows=1054
saved: data\processed\UD_Italian-ISDT\it_isdt-ud-dev.csv | rows=556
saved: data\processed\UD_Italian-ISDT\it_isdt-ud-test.csv | rows=487
saved: data\processed\UD_Italian-ISDT\it_isdt-ud-train.csv | rows=13020
saved: data\processed\UD_Italian-MarkIT\it_markit-ud-dev.csv | rows=310
saved: data\processed

In [8]:
summary_df = pd.DataFrame(summary_rows).sort_values(["dataset", "raw_file"]).reset_index(drop=True)
summary_df

,dataset,raw_file,processed_file,n_rows,n_positive,n_negative,clean_text_len,newline_only_boundaries
0,UD_English-EWT,en_ewt-ud-dev.sent_split,en_ewt-ud-dev.csv,2294,1494,800,128840,507
1,UD_English-EWT,en_ewt-ud-test.sent_split,en_ewt-ud-test.csv,2083,1443,640,128449,634
2,UD_English-EWT,en_ewt-ud-train.sent_split,en_ewt-ud-train.csv,13649,10164,3485,1025446,2380
3,UD_English-GUM,en_gum-ud-dev.sent_split,en_gum-ud-dev.csv,1568,1416,152,139536,159
4,UD_English-GUM,en_gum-ud-test.sent_split,en_gum-ud-test.csv,1420,1294,126,145820,170
5,UD_English-GUM,en_gum-ud-train.sent_split,en_gum-ud-train.csv,10020,9090,930,881738,1134
6,UD_English-PUD,en_pud-ud-test.sent_split,en_pud-ud-test.csv,1054,1000,54,112913,0
7,UD_English-ParTUT,en_partut-ud-dev.sent_split,en_partut-ud-dev.csv,153,147,6,14342,9
8,UD_English-ParTUT,en_partut-ud-test.sent_split,en_partut-ud-test.csv,152,151,1,18657,2
9,UD_English-ParTUT,en_partut-ud-train.sent_split,en_partut-ud-train.csv,1724,1661,63,236437,120


In [9]:
summary_path = PROCESSED_ROOT / "processing_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8")

print("saved summary:", summary_path)

saved summary: data\processed\processing_summary.csv


In [10]:
df_train = pd.read_csv("./data/processed/UD_English-EWT/en_ewt-ud-train.csv")
df_train.head()

,doc_id,char_idx,punct,left_context,right_context,centered_context,token_before,token_after,label
0,en_ewt-ud-train,128,.,"mosque in the town of Qaim, near the Syrian bo...",\n [This killing of a respected\ncleric will b...,"mosque in the town of Qaim, near the Syrian bo...",border,NaN,1
1,en_ewt-ud-train,211,.,leric will be causing us trouble for years to ...,]\n DPA: Iraqi authorities\nannounced that the...,leric will be causing us trouble for years to ...,come,NaN,1
2,en_ewt-ud-train,310,.,d busted up 3 terrorist cells operating in Bag...,\n Two of\nthem were being run by 2 officials ...,d busted up 3 terrorist cells operating in Bag...,Baghdad,Two,1
3,en_ewt-ud-train,386,!,run by 2 officials of the Ministry of the Inte...,\n The MoI in\nIraq is equivalent to the US FB...,run by 2 officials of the Ministry of the Inte...,Interior,The,1
4,en_ewt-ud-train,464,.,"lent to the US FBI, so this would be like havi...",Edgar Hoover\nunwittingly employ at a high le...,"lent to the US FBI, so this would be like havi...",J,Edgar,0


In [11]:
summary_df["positive_ratio"] = summary_df["n_positive"] / summary_df["n_rows"]
summary_df[["dataset", "raw_file", "n_rows", "n_positive", "n_negative", "positive_ratio"]]

,dataset,raw_file,n_rows,n_positive,n_negative,positive_ratio
0,UD_English-EWT,en_ewt-ud-dev.sent_split,2294,1494,800,0.651264
1,UD_English-EWT,en_ewt-ud-test.sent_split,2083,1443,640,0.692751
2,UD_English-EWT,en_ewt-ud-train.sent_split,13649,10164,3485,0.744670
3,UD_English-GUM,en_gum-ud-dev.sent_split,1568,1416,152,0.903061
4,UD_English-GUM,en_gum-ud-test.sent_split,1420,1294,126,0.911268
5,UD_English-GUM,en_gum-ud-train.sent_split,10020,9090,930,0.907186
6,UD_English-PUD,en_pud-ud-test.sent_split,1054,1000,54,0.948767
7,UD_English-ParTUT,en_partut-ud-dev.sent_split,153,147,6,0.960784
8,UD_English-ParTUT,en_partut-ud-test.sent_split,152,151,1,0.993421
9,UD_English-ParTUT,en_partut-ud-train.sent_split,1724,1661,63,0.963457
